# ARC-AGI-3 STaR LoRA — Colab QLoRA training (A100)

Runs the same train_lora.py as the Kaggle kernel but in QLORA mode
(4-bit NF4 base). On A100-80GB measured ~236s/step -> ~9-10h for
SUBSAMPLE=1200, 1 epoch (hybrid delta-net layers take the unfused
PyTorch path; flash-linear-attention not installed).

**Runtime: pick A100 (high-RAM) via Runtime > Change runtime type.**
L4-24GB likely OOMs at seq 8192 (vocab 248k logits); if stuck on L4,
set MAX_LEN to 4096 below (halves logit memory).

**Versions matter:** qwen3_5 needs transformers>=5.x AND peft>=0.21
(peft 0.17 imports HybridCache, removed in transformers 5.x).
train_lora.py filters TrainingArguments kwargs by signature, so it
runs on both 4.x and 5.x.

**Kaggle auth:** cell 2 writes ~/.kaggle/kaggle.json from the
`KAGGLE_KEY` Colab Secret. That key MUST belong to the Kaggle user
that owns takumuhata/taaf-duck-sft-v1 — a different account's key
gets 403 on the private dataset. If that happens, download the
dataset elsewhere and upload the files via the left Files panel
(session storage), then skip the sft download in cell 3. The same
goes for cell 5's upload — alternatively pull /content/lora_adapter
down through the Files panel and push the dataset locally.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       'peft==0.21.0', 'accelerate>=1.6', 'datasets>=3.0',
                       'transformers>=5.10.0', 'bitsandbytes>=0.45.0',
                       'kaggle==1.7.4.5'])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'pick a GPU runtime (A100 recommended)'

In [ ]:
import json, os, pathlib, subprocess
from google.colab import userdata
kd = pathlib.Path.home() / '.kaggle'; kd.mkdir(exist_ok=True)
kj = kd / 'kaggle.json'
kj.write_text(json.dumps({'username': 'takumuhata',
                          'key': userdata.get('KAGGLE_KEY')}))
os.chmod(kj, 0o600)
r = subprocess.run(['kaggle', 'datasets', 'list', '-m'],
                   capture_output=True, text=True)
print((r.stdout or r.stderr)[:400])

In [ ]:
# NOTE: a KAGGLE_KEY belonging to a *different* kaggle user gets
# 403 on the private takumuhata/ datasets. If dl() raises HTTPError
# on taaf-duck-sft-v1, upload packed_sft.jsonl + train_lora.py via
# the left Files panel into /content/sft_data instead.
import os, pathlib, requests, zipfile, json
BASE = pathlib.Path('/content/base_27b'); BASE.mkdir(exist_ok=True)
SFTD = pathlib.Path('/content/sft_data'); SFTD.mkdir(exist_ok=True)
KEY = json.loads((pathlib.Path.home()/'.kaggle'/'kaggle.json').read_text())['key']

def dl(slug, dest, marker):
    if (dest / marker).exists():
        print(slug, 'already present'); return
    zp = pathlib.Path('/content') / (slug.split('/')[-1] + '.zip')
    print('downloading', slug, '->', zp, flush=True)
    url = 'https://www.kaggle.com/api/v1/datasets/download/' + slug
    with requests.get(url, auth=('takumuhata', KEY), stream=True,
                      allow_redirects=True) as r:
        r.raise_for_status()
        with open(zp, 'wb') as f:
            n = 0
            for chunk in r.iter_content(1 << 20):
                if chunk:
                    f.write(chunk); n += len(chunk)
            print('downloaded', n / 1e9, 'GB')
    print('unzipping...')
    with zipfile.ZipFile(zp) as z:
        z.extractall(dest)
    zp.unlink()

dl('rahim3/qwen3-8-27b-bf16', BASE, 'config.json')
dl('takumuhata/taaf-duck-sft-v1', SFTD, 'packed_sft.jsonl')
print('base files:', len(list(BASE.glob('*.safetensors'))), 'shards')
print('sft files:', sorted(p.name for p in SFTD.iterdir()))

In [ ]:
import os, subprocess, sys
env = dict(os.environ)
env.update({
    'BASE_MODEL': '/content/base_27b',
    'SFT_JSONL': '/content/sft_data/packed_sft.jsonl',
    'OUT_DIR': '/content/lora_adapter',
    'EPOCHS': '1',
    'SUBSAMPLE': '1200',
    'LR': '1e-4',
    'MAX_LEN': '8192',
    'QLORA': '1',
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
})
train = '/content/sft_data/train_lora.py'
assert os.path.exists(train)
proc = subprocess.run([sys.executable, train], env=env, text=True)
assert proc.returncode == 0, 'train_lora failed'
import pathlib
print('adapter files:', sorted(p.name for p in
      pathlib.Path('/content/lora_adapter').iterdir()))

In [ ]:
# If this 403s (KAGGLE_KEY is not the dataset owner), pull
# /content/lora_adapter down via the Files panel and run
# `kaggle datasets version -p lora_adapter` locally instead.
import json, pathlib, subprocess
ad = pathlib.Path('/content/lora_adapter')
assert (ad / 'adapter_config.json').exists() and (ad / 'adapter_model.safetensors').exists(), 'adapter incomplete'
(ad / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'taaf-duck-lora-v1', 'id': 'takumuhata/taaf-duck-lora-v1',
    'licenses': [{'name': 'other'}]}))
r = subprocess.run(['kaggle', 'datasets', 'version', '-p', str(ad),
                    '-m', 'qlora adapter (colab a100)'], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
if r.returncode != 0:
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(ad)],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    assert r.returncode == 0, 'dataset upload failed'
print('uploaded -> takumuhata/taaf-duck-lora-v1')
print('next: push submit_ag3_lora on kaggle (still needs ~7h kaggle GPU)')